# Modelling - Quick Implementation

In [1]:
from databricks.connect import DatabricksSession
from pyspark.sql import functions
from pyspark.sql.window import Window
spark = DatabricksSession.builder.getOrCreate()

## Feature Engineering
- Starting off with 5 simple features that can be traced to domain experience and eda done. They include: `Type_risk`, `Power`, `Value_vehicle`, `Seniority`, `Payment`
- For each of these features - implement the transforms for each e.g value vehicle - log transform it?
- Derive the exposure here - eda reveals its 1 year for policy period and the targets (`severity`) as `silver_ml` layer
- Rename the target to make it clear to identify the targets
- The idea here is to create the train, dev, test split, maintain the porportions in all the target variables
- Do some sense checks e.g no id not duplicated in different split, no leakage rows etc.

In [5]:
CATALOG = "commercial-insurance-underwriting-solution-catalog"
SEED = 42
policy = spark.table(f"`{CATALOG}`.silver.policy")
policy.show(5)

+---+-------------------+-----------------+-----------------+----------+--------------------+--------------------+---------+-----------------+------------+------------+-----+----------+-------+-------+----------------+-------------+----------------+----------------+---------+----+-------------+------------------+-----+-----------------+-------------+-------+---------+------+------+
| ID|Date_start_contract|Date_last_renewal|Date_next_renewal|Date_birth|Date_driving_licence|Distribution_channel|Seniority|Policies_in_force|Max_policies|Max_products|Lapse|Date_lapse|Payment|Premium|Cost_claims_year|N_claims_year|N_claims_history|R_Claims_history|Type_risk|Area|Second_driver|Year_matriculation|Power|Cylinder_capacity|Value_vehicle|N_doors|Type_fuel|Length|Weight|
+---+-------------------+-----------------+-----------------+----------+--------------------+--------------------+---------+-----------------+------------+------------+-----+----------+-------+-------+----------------+------------

In [17]:
feature_column_list = ['ID','Type_risk', 'Power', 'Value_vehicle', 'Seniority', 'Payment']
basic_features = (
    policy
    .select(feature_column_list)
    .withColumn('Value_vehicle_log', functions.log(functions.col('Value_vehicle').alias("Value_vehicle_log")))
    .withColumn('Exposure', functions.lit(1))
    .drop('Value_vehicle')
)

In [19]:
for column in basic_features.columns:
    basic_features = basic_features.withColumnRenamed(column, column.lower())
basic_features.show(5)

+---+---------+-----+---------+-------+-----------------+--------+
| id|type_risk|power|seniority|payment|value_vehicle_log|exposure|
+---+---------+-----+---------+-------+-----------------+--------+
|  1|        1|   80|        4|      0|8.863332833439587|       1|
|  1|        1|   80|        4|      0|8.863332833439587|       1|
|  1|        1|   80|        4|      0|8.863332833439587|       1|
|  1|        1|   80|        4|      0|8.863332833439587|       1|
|  2|        1|   80|        4|      1|8.863332833439587|       1|
+---+---------+-----+---------+-------+-----------------+--------+
only showing top 5 rows
